# API call chatbot

I build a chatbot, that I can use in command promt.

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())  # read local .env file

True

Let's start coding with Anthropic SDK

In [17]:
import anthropic

client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from the environment

MODEL = "claude-haiku-4-5"

In [18]:
reply = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "Give me 3 books"}
    ]
)

In [19]:
print(reply)

Message(id='msg_017iyXg73VksqycXqhpQ2aEK', container=None, content=[TextBlock(citations=None, text='# 3 Book Recommendations\n\n1. **"The Midnight Library" by Matt Haig**\n   - A thought-provoking novel about exploring alternate life paths and finding meaning\n\n2. **"Atomic Habits" by James Clear**\n   - A practical guide to building good habits and breaking bad ones through small, consistent changes\n\n3. **"Project Hail Mary" by Andy Weir**\n   - A gripping sci-fi adventure about an astronaut trying to save humanity with humor and ingenuity\n\nWould you like recommendations in a specific genre or style?', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=13, output_tokens=130, output_token

In [20]:
print(reply.content)

[TextBlock(citations=None, text='# 3 Book Recommendations\n\n1. **"The Midnight Library" by Matt Haig**\n   - A thought-provoking novel about exploring alternate life paths and finding meaning\n\n2. **"Atomic Habits" by James Clear**\n   - A practical guide to building good habits and breaking bad ones through small, consistent changes\n\n3. **"Project Hail Mary" by Andy Weir**\n   - A gripping sci-fi adventure about an astronaut trying to save humanity with humor and ingenuity\n\nWould you like recommendations in a specific genre or style?', type='text')]


In [21]:
print(reply.content[0])

TextBlock(citations=None, text='# 3 Book Recommendations\n\n1. **"The Midnight Library" by Matt Haig**\n   - A thought-provoking novel about exploring alternate life paths and finding meaning\n\n2. **"Atomic Habits" by James Clear**\n   - A practical guide to building good habits and breaking bad ones through small, consistent changes\n\n3. **"Project Hail Mary" by Andy Weir**\n   - A gripping sci-fi adventure about an astronaut trying to save humanity with humor and ingenuity\n\nWould you like recommendations in a specific genre or style?', type='text')


In [22]:
print(reply.content[0].text)

# 3 Book Recommendations

1. **"The Midnight Library" by Matt Haig**
   - A thought-provoking novel about exploring alternate life paths and finding meaning

2. **"Atomic Habits" by James Clear**
   - A practical guide to building good habits and breaking bad ones through small, consistent changes

3. **"Project Hail Mary" by Andy Weir**
   - A gripping sci-fi adventure about an astronaut trying to save humanity with humor and ingenuity

Would you like recommendations in a specific genre or style?


 We have new message every time the loop runs, so we can't keep track the message. It is more like you ask one question then it answer a one question but cannot answer the question that continue from the previous ones.

In [6]:
while True:
    
    user_input = input("User: ")
    
    if user_input.lower() in ["exit", "quit", "q"]:
        print("Exiting the chatbot. Goodbye!")
        break
    
    reply = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        messages=[
            {"role": "user", "content": user_input}
        ]
    )
    
    print("Bot: " + reply.content[0].text)
    
    print("\n-----------\n")

Bot: # Charles de Gaulle - Étoile to Eiffel Tower

**Shortest route:**

1. **Metro Line 6** from Charles de Gaulle - Étoile
2. Get off at **Bir-Hakeim** station
3. Walk ~5 minutes to Eiffel Tower

**Travel time:** ~15 minutes total

Alternatively, you can walk to nearby Metro stations and take Line 9 toward Pont de Sèvres, exiting at Trocadéro for scenic views of the tower.

-----------

Bot: # 3 Book Recommendations

1. **"Atomic Habits" by James Clear**
   - A practical guide to building good habits and breaking bad ones. Great for anyone looking to improve their daily routines and make lasting change.

2. **"Educated" by Tara Westover**
   - A gripping memoir about a woman raised by survivalists who educates herself and escapes her isolated upbringing. Thought-provoking and hard to put down.

3. **"Dune" by Frank Herbert**
   - An epic science fiction novel with complex world-building, politics, and adventure. Perfect if you want to immerse yourself in a richly detailed universe.

W

Loop that can keep track of the conversation by storing the messages in a list and appending new messages to it. This way, the chatbot can respond in context to previous messages.


In [ ]:
messages = []  # Initialize an empty list to store the conversation

while True:
    print("\n---------------")
    user_input = input("User: ")
    
    if user_input.lower() in ["exit", "quit", "q"]:
        print("Exiting the chatbot. Goodbye!")
        break
    
    # Append the user's message to the conversation
    messages.append({"role": "user", "content": user_input})
    
    # Get the chatbot's reply
    reply = client.messages.create(
        model = MODEL,
        max_tokens = 1024,
        messages = messages
    )
    
    print("Bot: " + reply.content[0].text)
    
    messages.append({"role": "assistant", "content": reply.content[0].text})
    
    print("---------------")
    print(reply.usage)
    

However doing this, I can see a problem that it spent a lot of tokens. So there are other ways to save the token by using cache, or summary the messages into 1 paragrah if possible after using it for 5 times.

```
reply = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=messages,
    cache_control={"type": "ephemeral"},   # <-- caches the reusable prefix automatically
)
```

I am going to use openai sk

In [23]:
import openai

client_2 = openai.OpenAI()  # reads OPENAI_API_KEY from the environment

model_openai = "gpt-4o-mini"

In [24]:
response = client_2.chat.completions.create(
    model=model_openai,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hello! Give me 3 books"}
    ]
)

print(response)

ChatCompletion(id='chatcmpl-DxB7c9YhaOTjwg4U9tIQIpXFShShL', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Sure! Here are three diverse book recommendations across different genres:\n\n1. **Fiction**: *The Night Circus* by Erin Morgenstern - This enchanting novel tells the story of a magical competition between two young illusionists, set in a mysterious circus that appears only at night. The rich prose and imaginative setting create a captivating reading experience.\n\n2. **Non-Fiction**: *Sapiens: A Brief History of Humankind* by Yuval Noah Harari - This thought-provoking book explores the history of our species from the emergence of Homo sapiens to the present day, weaving together anthropology, history, and economics to discuss how we have shaped the world.\n\n3. **Science Fiction**: *Dune* by Frank Herbert - A classic of the genre, *Dune* is set on the desert planet of Arrakis and follows the story of Paul Atreides, who becomes

In [15]:
print(response.choices[0].message.content)

Sure! Here are three diverse book recommendations across different genres:

1. **Fiction**: *Where the Crawdads Sing* by Delia Owens
   - A coming-of-age story set in the marshes of North Carolina, this novel intertwines a murder mystery with themes of loneliness and resilience. It follows Kya Clark, the "Marsh Girl," who grows up isolated from society.

2. **Non-Fiction**: *Educated* by Tara Westover
   - This memoir recounts Tara Westover's journey from growing up in a strict and abusive household in rural Idaho with no formal education to earning a PhD from Cambridge University. It explores themes of self-discovery, resilience, and the transformative power of education.

3. **Science Fiction**: *The Martian* by Andy Weir
   - This gripping novel tells the story of Mark Watney, an astronaut stranded on Mars who must use his ingenuity and determination to survive. It's both a thriller and an inspiring testament to human resourcefulness and science.

Let me know if you would like more 

Let's use ADK of google instead

In [4]:
from google import genai

client_3 = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))  # reads GOOGLE_API_KEY from the environment

model_google = "gemini-2.5-flash"

In [5]:
mess = client_3.models.generate_content(
    model=model_google,
    contents="Hello! Give me 3 books"
)

In [6]:
print(mess)

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text="""Okay, since you haven't given me any preferences, I'll give you a diverse mix that are generally well-loved!

1.  **Project Hail Mary** by Andy Weir
    *   **Why:** If you like science fiction, humor, and a thrilling mystery, this is a fantastic choice. It's about an astronaut who wakes up with amnesia on a mission to save humanity, and he has to piece together what's going on and solve incredibly complex problems with limited resources. It's smart, funny, and incredibly engaging.

2.  **Circe** by Madeline Miller
    *   **Why:** For those who enjoy mythology, beautiful prose, and strong character development. This book retells the story of Circe, the goddess of witchcraft from Greek mythology, exploring her exile and journey of self-discovery. It's lyrical, immersive, and gives a fresh perspective on ancient tales.

3.  **To Kill a Mockingbird**

In [8]:
print(mess.candidates[0])

content=Content(
  parts=[
    Part(
      text="""Okay, since you haven't given me any preferences, I'll give you a diverse mix that are generally well-loved!

1.  **Project Hail Mary** by Andy Weir
    *   **Why:** If you like science fiction, humor, and a thrilling mystery, this is a fantastic choice. It's about an astronaut who wakes up with amnesia on a mission to save humanity, and he has to piece together what's going on and solve incredibly complex problems with limited resources. It's smart, funny, and incredibly engaging.

2.  **Circe** by Madeline Miller
    *   **Why:** For those who enjoy mythology, beautiful prose, and strong character development. This book retells the story of Circe, the goddess of witchcraft from Greek mythology, exploring her exile and journey of self-discovery. It's lyrical, immersive, and gives a fresh perspective on ancient tales.

3.  **To Kill a Mockingbird** by Harper Lee
    *   **Why:** A timeless classic for a reason. Set in the American Sout

In [12]:
print(mess.candidates[0].content.parts)

[Part(
  text="""Okay, since you haven't given me any preferences, I'll give you a diverse mix that are generally well-loved!

1.  **Project Hail Mary** by Andy Weir
    *   **Why:** If you like science fiction, humor, and a thrilling mystery, this is a fantastic choice. It's about an astronaut who wakes up with amnesia on a mission to save humanity, and he has to piece together what's going on and solve incredibly complex problems with limited resources. It's smart, funny, and incredibly engaging.

2.  **Circe** by Madeline Miller
    *   **Why:** For those who enjoy mythology, beautiful prose, and strong character development. This book retells the story of Circe, the goddess of witchcraft from Greek mythology, exploring her exile and journey of self-discovery. It's lyrical, immersive, and gives a fresh perspective on ancient tales.

3.  **To Kill a Mockingbird** by Harper Lee
    *   **Why:** A timeless classic for a reason. Set in the American South during the Great Depression, it'

In [15]:
print(mess.parts[0].text)

Okay, since you haven't given me any preferences, I'll give you a diverse mix that are generally well-loved!

1.  **Project Hail Mary** by Andy Weir
    *   **Why:** If you like science fiction, humor, and a thrilling mystery, this is a fantastic choice. It's about an astronaut who wakes up with amnesia on a mission to save humanity, and he has to piece together what's going on and solve incredibly complex problems with limited resources. It's smart, funny, and incredibly engaging.

2.  **Circe** by Madeline Miller
    *   **Why:** For those who enjoy mythology, beautiful prose, and strong character development. This book retells the story of Circe, the goddess of witchcraft from Greek mythology, exploring her exile and journey of self-discovery. It's lyrical, immersive, and gives a fresh perspective on ancient tales.

3.  **To Kill a Mockingbird** by Harper Lee
    *   **Why:** A timeless classic for a reason. Set in the American South during the Great Depression, it's told through th